# Load a dataset

In [31]:
from datasets import load_dataset

dataset = load_dataset("imdb")
print(dataset)
print(type(dataset))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
<class 'datasets.dataset_dict.DatasetDict'>


In [32]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


# Load and tokenize a dataset (use training data)

In [37]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("imdb", split="train[:1000]")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

def tokenize(batch):
    return tokenizer(
        batch["text"],        # Input text column from dataset
        padding=True,         # Pad all sequences in the batch to the same length
        truncation=True       # Truncate sequences that are longer than model's max length
    )
    
dataset = dataset.map(tokenize, batched=True)

In [38]:
print(dataset)

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask'],
    num_rows: 1000
})


In [41]:
print(dataset[0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

# Describe the dataset

In [42]:
print(dataset.features)
print(dataset.num_rows) #len(dataset)
print(dataset.column_names)
dataset.shape

{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos']), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}
1000
['text', 'label', 'input_ids', 'attention_mask']


(1000, 4)

In [43]:
sliced = dataset.select(range(2))
print(sliced['label'])

Column([0, 0])


In [44]:
print(sliced[0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [47]:
positive = dataset.filter(lambda x: x["label"] == 1) #give rows where label is 1
negative = dataset.filter(lambda x: x["label"] == 0) #give rows where label is 0
print(len(positive))
print(len(negative))

0
1000


In [48]:
#First 1000 rows happeend to be all negative. How to select better

In [49]:
dataset = load_dataset("imdb", split="train")

# Shuffle the dataset randomly
dataset = dataset.shuffle(seed=42)  # seed ensures reproducibility

# Select first 1000 rows after shuffling
dataset = dataset.select(range(1000))

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

# Tokenization function
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding=True,
        truncation=True
    )

# Apply tokenization
dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [50]:
positive = dataset.filter(lambda x: x["label"] == 1) #give rows where label is 1
negative = dataset.filter(lambda x: x["label"] == 0) #give rows where label is 0
print(len(positive))
print(len(negative))

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

488
512


## Describe() like stats

In [51]:
import numpy as np

lengths = [len(x) for x in dataset["text"]]

print("Avg length:", np.mean(lengths))
print("Max length:", np.max(lengths))
print("Min length:", np.min(lengths))

Avg length: 1313.345
Max length: 5867
Min length: 174


## Pandas

In [52]:
df = dataset.to_pandas()

print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   text            1000 non-null   object
 1   label           1000 non-null   int64 
 2   input_ids       1000 non-null   object
 3   attention_mask  1000 non-null   object
dtypes: int64(1), object(3)
memory usage: 31.4+ KB
None
             label
count  1000.000000
mean      0.488000
std       0.500106
min       0.000000
25%       0.000000
50%       0.000000
75%       1.000000
max       1.000000


In [53]:
df.head()

,text,label,input_ids,attention_mask
0,There is no relation at all between Fortier an...,1,"[3862, 374, 902, 12687, 518, 678, 1948, 11002,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,This movie is a great. The plot is very true t...,1,"[1986, 5700, 374, 264, 2244, 13, 576, 7089, 37...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,"George P. Cosmatos' ""Rambo: First Blood Part I...",0,"[38952, 393, 13, 18114, 8470, 436, 6, 330, 638...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,In the process of trying to establish the audi...,1,"[641, 279, 1882, 315, 4460, 311, 5695, 279, 29...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,"Yeh, I know -- you're quivering with excitemen...",0,"[56, 2636, 11, 358, 1414, 1177, 498, 2299, 922...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [54]:
print(dataset[0].keys())

dict_keys(['text', 'label', 'input_ids', 'attention_mask'])


In [56]:
#dataset.remove_columns(["text"])
dataset.rename_column("label", "target") #rename label to target

Dataset({
    features: ['text', 'target', 'input_ids', 'attention_mask'],
    num_rows: 1000
})

In [60]:
sliced = dataset.select(range(2))
print(sliced.column_names)

['text', 'label', 'input_ids', 'attention_mask']


In [62]:
sliced.add_column("length", [10, 20])

Dataset({
    features: ['text', 'label', 'input_ids', 'attention_mask', 'length'],
    num_rows: 2
})

In [65]:
print(sliced["label"])

Column([1, 1])


In [69]:
dataset.unique("label")

[1, 0]

# Filter for search

In [70]:
long_reviews = dataset.filter(lambda x: len(x["text"]) > 500)

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [71]:
len(long_reviews)

898

In [75]:
import time

In [96]:
start_time = time.time()
great_positive = dataset.filter(
    lambda x: x["label"] == 1 and "great" in x["text"].lower()
)
end_time = time.time()

In [97]:
print(len(great_positive))
print(end_time - start_time)

166
0.0073010921478271484


In [98]:
#batchedd filtering

In [99]:
def batch_filter(batch):
    return [label == 1 and "great" in text.lower() for label, text in zip(batch["label"], batch["text"])]

In [100]:
start_time = time.time()
great_positive_batched = dataset.filter(batch_filter, batched=True, batch_size=1000)
end_time = time.time()

In [101]:
print(len(great_positive_batched))
print(end_time - start_time)

166
0.008153676986694336


In [102]:
#Batch processes all the rows using the lambda might look at first few rows. Hence batch slower